# TDE: Explorando Dados na Web com Python

**Contexto:** coleta de dados reais na web, do consumo de APIs REST até webscraping ético de páginas HTML.

Conteúdo abordado:
- Nível 1 (Básico): requisições GET, parâmetros, cabeçalhos e status HTTP
- Nível 2 (Intermediário): JSON, tratamento de erros e download de arquivos binários
- Nível 3 (Avançado): webscraping com BeautifulSoup, ética (`robots.txt`) e `pandas.read_html()`

Todas as requisições usam `timeout` para não travar o notebook indefinidamente e todos os arquivos gerados (imagem, CSVs) ficam nesta mesma pasta.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import io

## Nível 1 (Básico): URLs, Parâmetros e Cabeçalhos

**Exercício 1.1:** requisição GET para a API pública do JSONPlaceholder.
**Exercício 1.2:** uso de `params` para filtrar, trazendo só os posts do `userId` 2.
**Exercício 1.3:** cabeçalho `User-Agent` personalizado.
**Exercício 1.4:** impressão do `status_code` e da `url` final montada pelo `requests`, sempre com `timeout`.

In [ ]:
headers = {'User-Agent': 'TDE-WebPython-FaculdadeMarcell/1.0'}
params = {'userId': 2}

resposta = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params=params,
    headers=headers,
    timeout=10,
)

print('Status code:', resposta.status_code)
print('URL final:', resposta.url)
print('Quantidade de posts retornados:', len(resposta.json()))

## Nível 2 (Intermediário): JSON, Erros e Arquivos Binários

**Exercício 2.1:** consulta de três CEPs na API do ViaCEP, convertendo cada resposta com `.json()` e reunindo tudo num DataFrame.

In [ ]:
ceps = ['01001000', '20040020', '70150900']
enderecos = []

for cep in ceps:
    resposta = requests.get(f'https://viacep.com.br/ws/{cep}/json/', timeout=10)
    enderecos.append(resposta.json())

df_ceps = pd.DataFrame(enderecos)
df_ceps[['cep', 'logradouro', 'bairro', 'localidade', 'uf']]

**Exercício 2.2:** função de download segura, com `try/except` e `resposta.raise_for_status()` para capturar erros HTTP (404, 500, etc.) e avisar de forma amigável.

In [ ]:
def baixar_arquivo(url, caminho_arquivo, timeout=10):
    try:
        resposta = requests.get(url, timeout=timeout)
        resposta.raise_for_status()
    except requests.exceptions.HTTPError:
        print(f'Falha ao baixar {url}: o servidor respondeu com erro {resposta.status_code}')
        return False
    except requests.exceptions.RequestException as erro:
        print(f'Falha ao baixar {url}: {erro}')
        return False

    with open(caminho_arquivo, 'wb') as f:
        f.write(resposta.content)

    print(f'Arquivo salvo em {caminho_arquivo}')
    return True


# demonstrando a captura de erro com uma URL inexistente na mesma API
baixar_arquivo('https://jsonplaceholder.typicode.com/posts/99999999', 'nao_deve_existir.bin')

**Exercício 2.3:** download de uma imagem aleatória do Picsum, salvando o conteúdo bruto (`resposta.content`) em modo binário (`'wb'`), usando a mesma função `baixar_arquivo`.

In [ ]:
baixar_arquivo('https://picsum.photos/400/400', 'imagem_aleatoria.jpg')

## Nível 3 (Avançado): Webscraping e Ética

**Exercício 3.1:** antes de raspar o site, checar o `robots.txt` para verificar o que é permitido. Se o arquivo não existir (404), significa que o site não publicou restrições explícitas, mas a coleta deve continuar comedida e respeitosa mesmo assim.

In [ ]:
resposta_robots = requests.get('https://books.toscrape.com/robots.txt', timeout=10)

if resposta_robots.status_code == 200:
    print('robots.txt encontrado:')
    print(resposta_robots.text)
else:
    print(f'robots.txt retornou status {resposta_robots.status_code} (sem restrições explícitas publicadas).')
    print('Mesmo assim, a coleta seguirá comedida: poucas páginas, com timeout e sem sobrecarregar o servidor.')

**Exercício 3.2:** raspagem dos 5 primeiros livros de `books.toscrape.com` (site feito para prática de scraping) com BeautifulSoup (`'html.parser'`), extraindo título e preço.

In [ ]:
resposta_livros = requests.get('https://books.toscrape.com/', timeout=10)
# usa .content (bytes) em vez de .text: o site nao declara charset no header
# Content-Type, entao requests cai no default ISO-8859-1 e corrompe o "£";
# passando bytes, o BeautifulSoup detecta o encoding real (utf-8) sozinho
sopa = BeautifulSoup(resposta_livros.content, 'html.parser')

livros = sopa.select('article.product_pod')[:5]

dados_livros = []
for livro in livros:
    titulo = livro.h3.a['title']
    preco = livro.find('p', class_='price_color').text
    dados_livros.append({'titulo': titulo, 'preco': preco})

df_livros = pd.DataFrame(dados_livros)
df_livros

**Exercício 3.3:** dados extraídos salvos em CSV com Pandas.

In [ ]:
df_livros.to_csv('livros_scraping.csv', index=False, encoding='utf-8')
print('livros_scraping.csv salvo')

**Exercício 3.4:** tabela de uma página da Wikipedia (lista de países por população) capturada direto com `pandas.read_html()` combinado com `io.StringIO()`, sem precisar do BeautifulSoup.

In [ ]:
resposta_wiki = requests.get(
    'https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o',
    headers=headers,
    timeout=10,
)

tabelas = pd.read_html(io.StringIO(resposta_wiki.text))
print(f'{len(tabelas)} tabelas encontradas na página')

df_populacao = tabelas[0]
df_populacao.head(10)